In [ ]:
import json
import numpy as np
import sys
sys.path.append("../")
import pandas as pd
import argparse
import json
import os
import sys

import numpy as np
import torch

sys.path.insert(0, ".")
from config.config import Config
from model.schedule import get_schedule
from properties.steering import (load_residual_stats, sweep_candidates,
                                  select_operating_point, heatmap_data,
                                  plot_heatmap, reference_fingerprints,
                                  DirectionArtifact, DirectionStore)

In [ ]:
import sys
sys.path.append("../datasets")
from tokenizer import SelfiesTokenizer
from model.model import MDLM


In [ ]:
artifacts = "../artifacts/p2"
property = "logp"
alpha_ref= 2.0
site = "pooled"
estimator = "diffmeans"
data = "../artifacts/processed"
device = "cuda"
ckpt = r"C:\Users\tsuma.thomas\Documents\MARS\runs\phase1-configb\ckpt_0020000.pt"

In [ ]:
def load_model(ckpt, cfg, tok, device, use_ema=True):
    model = MDLM(tok.vocab_size, cfg.model, pad_id=tok.pad_id).to(device)
    ck = torch.load(ckpt, map_location=device)
    model.load_state_dict(ck["model"])
    if use_ema and "ema" in ck:
        sd = model.state_dict()
        for k, v in ck["ema"].items():
            sd[k].copy_(v.to(sd[k].dtype))
    model.eval()
    return model, ck.get("step", -1)

In [ ]:
cfg_path = os.path.join(os.path.dirname(ckpt), "config.json")
cfg = Config.from_json(cfg_path)
tok = SelfiesTokenizer.load(os.path.join(data, "tokenizer.json"))
schedule = get_schedule(cfg.diffusion.schedule)
model, mstep = load_model(ckpt, cfg, tok, device)


In [ ]:
df = pd.read_csv(r"C:\Users\tsuma.thomas\Documents\MARS\artifacts\p2\sweep_logp.csv")

In [ ]:
df

,layer,position,prop_mean,delta_property,fcd_proxy,heavy_shift,n_valid,skipped
0,0,0,2.976900,1.046729,0.860895,-5.147507,499,False
1,1,0,1.559599,-0.370572,0.846220,-6.681726,497,False
2,2,0,1.567354,-0.362817,0.845785,-6.512712,497,False
3,3,0,1.527153,-0.403019,0.846176,-6.396012,497,False
4,4,0,1.492666,-0.437505,0.848527,-6.542893,497,False
5,5,0,1.489717,-0.440454,0.848233,-6.432229,497,False
6,6,0,1.491950,-0.438222,0.851121,-7.106274,497,False
7,7,0,1.498637,-0.431535,0.851393,-7.116334,497,False
8,8,0,1.395215,-0.534956,0.854495,-7.210901,497,False
9,9,0,1.319781,-0.610390,0.857844,-7.601243,497,False


In [ ]:
ok = df[~df["skipped"].astype(bool) & (df["n_valid"] > 0)].copy()

In [ ]:
max(ok["fcd_proxy"].abs())

0.8610313184272098

In [ ]:
ok

,layer,position,prop_mean,delta_property,fcd_proxy,heavy_shift,n_valid,skipped


In [ ]:
ok

,layer,position,prop_mean,delta_property,fcd_proxy,heavy_shift,n_valid,skipped
0,0,0,2.976900,1.046729,0.860895,-5.147507,499,False
1,1,0,1.559599,-0.370572,0.846220,-6.681726,497,False
2,2,0,1.567354,-0.362817,0.845785,-6.512712,497,False
3,3,0,1.527153,-0.403019,0.846176,-6.396012,497,False
4,4,0,1.492666,-0.437505,0.848527,-6.542893,497,False
5,5,0,1.489717,-0.440454,0.848233,-6.432229,497,False
6,6,0,1.491950,-0.438222,0.851121,-7.106274,497,False
7,7,0,1.498637,-0.431535,0.851393,-7.116334,497,False
8,8,0,1.395215,-0.534956,0.854495,-7.210901,497,False
9,9,0,1.319781,-0.610390,0.857844,-7.601243,497,False


In [ ]:
best = select_operating_point(df, 1,
                                  10
                            )

  selected layer 0 position 0: Δ=+1.047  fcd~0.861  heavy_shift=-5.15


In [ ]:
best

{'layer': 0,
 'position': 0,
 'delta_property': 1.0467286001603209,
 'fcd_proxy': 0.8608946909115728,
 'heavy_shift': -5.147507014028054}

In [ ]:
directions = torch.from_numpy(np.load(r"C:\Users\tsuma.thomas\Documents\MARS\artifacts\p2\directions_logp_pooled.npz")['diffmeans'])

In [43]:
n_layers, n_sites, _ = directions.shape
print(n_layers)
print(n_sites)
print(_)

15
1
768


In [47]:
Z = heatmap_data(df, n_layers, n_sites)
labels = (["BOS"] + [f"P{i}" for i in range(n_sites - 1)]
          if site == "prefix" and n_sites > 1 else None)
fig = plot_heatmap(Z, os.path.join(artifacts,
                                   f"figA_{property}.png"),
                   title=f"Layer x position sensitivity — {property} "
                         f"(site: {site}, alpha={alpha_ref})",
                   selected=best, position_labels=labels)
print(f"[select] Figure A -> {fig}")

[select] Figure A -> ./artifacts/p2\figA_logp.png


In [48]:
fig

'./artifacts/p2\\figA_logp.png'

In [53]:
with open(os.path.join(artifacts, f"probe_report_{property}.json")) as f:
        report = json.load(f)
proj = next((r["projection_spearman"] for r in report["projection_checks"]
                 if r["estimator"] == estimator), float("nan"))

In [72]:
art = DirectionArtifact(
    vector=directions[best["layer"], best["position"]].numpy(),
    property=property, layer=best["layer"], site=site,
    position=best["position"], estimator=estimator,
    corpus=os.path.basename(data), split="train",
    n_samples=report["gate_a_stats"].get("n_positive", 0),
    seed=42, projection_spearman=proj,
    heavy_mean_abs_diff=report["gate_a_stats"].get("heavy_mean_abs_diff", float("nan")),
    model_ckpt=ckpt, model_step=mstep,
    extra={"alpha_ref": alpha_ref, "delta_property": best["delta_property"]})
store = DirectionStore(os.path.join(artifacts, "directions"))
aid = store.save(art)

In [77]:
with open(os.path.join(artifacts, f"selected_{property}.json"), "w") as f:
        json.dump({"direction_id": aid, "site": site, "estimator": estimator,
                    **best}, f, indent=2)
print("\n-> next: scripts/p2_alpha_sweep.py (E2, the headline Pareto frontier)")


-> next: scripts/p2_alpha_sweep.py (E2, the headline Pareto frontier)
